# Predictive Classification using Random Forest Ensembles
**Name:** Poorvi M S  
**Batch:** Artificial Intelligence – September 2026

## Project objective
Build a Random Forest classifier for the Wisconsin Diagnostic Breast Cancer dataset to predict whether a tumour is **benign** or **malignant**. This notebook follows the provided project brief: exploratory analysis, missing-value handling, categorical encoding where applicable, an 80:20 split, Random Forest training, and evaluation with a confusion matrix, precision, recall, F1-score, ROC-AUC, and feature importance.

**Dataset:** [Kaggle – Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)



## 1. Load the dataset
Download `data.csv` from Kaggle and upload it when prompted. The original Kaggle file typically contains an `id` column, a `diagnosis` target (`M` / `B`), 30 numeric measurement columns, and an `Unnamed: 32` column that is empty. This cell also supports common capitalization variants.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

df = pd.read_csv("/content/data.csv")
df.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## 2. Initial inspection and data quality
Inspect column types, missing values, and duplicate records before modeling. The ID column does not describe tumour characteristics and is removed. An entirely empty column such as `Unnamed: 32` is also removed.

In [ ]:
print("Column information:")
display(df.dtypes.to_frame("data_type"))
print("\\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
print("\\nDuplicate rows:", df.duplicated().sum())

# Find the target column case-insensitively.
target_candidates = [c for c in df.columns if c.lower() in ["diagnosis", "target", "class"]]
if not target_candidates:
    raise ValueError("Could not find the target column. Expected a 'diagnosis' column.")
target_col = target_candidates[0]

# Remove columns that are IDs or completely empty.
drop_cols = [c for c in df.columns if c.lower() in ["id", "unnamed: 32"] or df[c].isna().all()]
df = df.drop(columns=drop_cols, errors="ignore")
print("Removed columns:", drop_cols)

## 3. Exploratory data analysis (EDA)
The class distribution chart shows whether the target labels are balanced. The correlation heatmap helps identify groups of strongly related numerical measurements. Correlation does not prove causation and is not, by itself, a reason to remove a feature from a Random Forest.

In [ ]:
# Normalize the target labels to readable values.
df[target_col] = df[target_col].astype(str).str.strip()
label_map = {"M": "Malignant", "B": "Benign", "1": "Malignant", "0": "Benign"}
df[target_col] = df[target_col].replace(label_map)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Diagnosis Class Distribution")
plt.xlabel("Diagnosis"); plt.ylabel("Number of records")
plt.tight_layout(); plt.show()

numeric_for_corr = df.drop(columns=[target_col]).select_dtypes(include=np.number)
plt.figure(figsize=(12, 9))
sns.heatmap(numeric_for_corr.corr(), cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

display(df.describe(include="all").T.head(15))

## 4. Preprocessing and train-test split
The target column is separated from the predictors. Non-numeric predictors, if present, are one-hot encoded. Missing numeric values are replaced with the median and missing categorical values with the most frequent value. The split is 80% training and 20% testing, with stratification to preserve the class proportions.

**Scaling decision:** StandardScaler is skipped because Random Forest uses decision-tree splits and is not distance-based; numeric scaling is generally unnecessary for this model.

In [ ]:
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Remove any leftover non-feature ID-like columns if present.
X = X.drop(columns=[c for c in X.columns if c.lower() == "id"], errors="ignore")

# Convert categorical predictors to indicator columns (one-hot encoding).
X = pd.get_dummies(X, drop_first=False)

# Ensure predictors are numeric and handle missing/infinite values.
X = X.replace([np.inf, -np.inf], np.nan)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# Define the positive class as Malignant for precision, recall and ROC-AUC.
positive_label = "Malignant"
if positive_label not in set(y):
    raise ValueError(f"Expected 'Malignant' in target labels, found: {sorted(map(str, y.unique()))}")

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))
print("Training class proportions:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 5. Train the Random Forest model
- `n_estimators=200`: averages predictions across many trees, usually improving stability over a very small forest at a moderate computational cost.
- `max_depth=10`: limits tree complexity to reduce the risk of overfitting while retaining useful interactions.
- `class_weight='balanced'`: adjusts class weights inversely to class frequency, which can help when class counts differ.
- `random_state=42`: makes the train/test split and model behavior reproducible.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

## 6. Evaluation metrics
Precision indicates how many cases predicted malignant were actually malignant. Recall indicates how many actual malignant cases were identified. For screening-like use cases, low recall would mean more malignant cases were missed. F1-score balances precision and recall. ROC-AUC measures how well the predicted scores rank malignant cases above benign cases across thresholds.

The positive class is **Malignant**. The confusion matrix is displayed with Malignant first, so the upper-left cell is true positives and the lower-right cell is true negatives.

In [ ]:
labels_order = ["Malignant", "Benign"]
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix (Malignant is positive)")
plt.tight_layout(); plt.show()

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=positive_label, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=positive_label, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=positive_label, zero_division=0)

positive_index = list(model.classes_).index(positive_label)
y_score = model.predict_proba(X_test)[:, positive_index]
y_binary = (y_test == positive_label).astype(int)
auc = roc_auc_score(y_binary, y_score)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision (Malignant): {precision:.4f}")
print(f"Recall (Malignant)   : {recall:.4f}")
print(f"F1-score (Malignant) : {f1:.4f}")
print(f"ROC-AUC              : {auc:.4f}")

## 7. ROC curve
The ROC curve plots true positive rate against false positive rate as the decision threshold changes. The diagonal line is the chance baseline; a curve closer to the top-left generally indicates stronger separation.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_binary, y_score)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance baseline")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 8. Feature importance
The chart ranks features by the Random Forest's impurity-based importance. Larger values mean the feature contributed more to splits across the fitted forest; this is not proof of causation, and correlated variables can share or distort importance.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top_features = importances.head(15).sort_values()
plt.figure(figsize=(9, 6))
top_features.plot(kind="barh")
plt.xlabel("Feature importance"); plt.title("Top 15 Feature Importances")
plt.tight_layout(); plt.show()
display(importances.head(15).to_frame("importance"))

## 9. Findings and conclusion
Review the printed scores and charts rather than relying on accuracy alone. If recall for Malignant is low, the model is missing too many actual malignant examples; if precision is low, more benign examples are incorrectly flagged as malignant. The F1-score summarizes the precision-recall balance, while ROC-AUC measures ranking performance across thresholds.

Random Forest is useful because it combines many decision trees and can model nonlinear relationships. Limitations include possible overfitting, bias in impurity-based importance, sensitivity to dataset representativeness, and the fact that this dataset is not a substitute for clinical validation. Possible improvements include cross-validation, systematic hyperparameter tuning, comparing Gradient Boosting, and checking calibration and subgroup performance.

**Submission checklist**
- Include this well-commented notebook with outputs after running all cells.
- Include a report describing data cleaning, the workflow, findings, and the strengths/weaknesses of the model.
- Include high-resolution class-distribution and correlation plots, confusion matrix, ROC curve with AUC, and feature-importance chart.
- Ensure metric values in the report match the notebook output after running it on the downloaded Kaggle CSV.